## Setup 
---

In [1]:
import os, re, string, warnings, time
from collections import Counter
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.sparse import hstack
from scipy.special import softmax
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.model_selection import GroupKFold
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import LinearSVC
from lightgbm import LGBMClassifier

SEED=42
N_FOLDS=int(os.environ.get('NTLLM_FOLDS', '5'))

# If doing an initial debugging run, set NTLLM_FOLDS=3. Final: use 5.
BASE_DIR = Path('/kaggle/input/competitions/name-that-llm')
assert (BASE_DIR/'train.csv').exists(), f'Missing official competition input: {BASE_DIR}'

train = pd.read_csv(BASE_DIR/'train.csv')
test  = pd.read_csv(BASE_DIR/'test.csv')

assert {'id','prompt_id','text','model'} <= set(train.columns)
assert {'id','text'} <= set(test.columns)
assert train.id.is_unique and test.id.is_unique and test.id.notna().all()
assert train.prompt_id.notna().all() and train.model.notna().all()
texts=train.text.fillna('').astype(str).tolist()
test_texts=test.text.fillna('').astype(str).tolist()
le=LabelEncoder().fit(train.model)
y=le.transform(train.model)
CLASSES=le.classes_.tolist()
K=len(CLASSES)
assert K == 18, f'Expected 18 labels; got {K}'
assert train.prompt_id.nunique() >= N_FOLDS
print('Train',train.shape,'test',test.shape,'prompt groups',train.prompt_id.nunique())
print('Model counts:',train.model.value_counts().to_dict())


Train (2699, 4) test (3, 2) prompt groups 150
Model counts: {'qwen3.8-max': 150, 'claude-sonnet-5': 150, 'gemini-3.6-flash': 150, 'gpt-5.6-terra': 150, 'glm-5.3-flash': 150, 'gemini-3.1-pro-preview': 150, 'gemini-3.8-flash': 150, 'claude-opus-4-6': 150, 'gpt-5.6-luna': 150, 'claude-opus-4-8': 150, 'gemini-3.7-flash': 150, 'claude-opus-5': 150, 'kimi-k3': 150, 'gpt-5.5': 150, 'glm-5.3': 150, 'gpt-6-astra': 150, 'gpt-5.6-sol': 150, 'claude-fable-5': 149}


## Orthographic features
---

In [2]:
FUNCTION_WORDS = set((
    'a an the and or but if while although because as so than that which who whom whose this these those '
    'it its they them their theirs he him his she her hers we us our ours you your yours '
    'i me my mine in on at to from with without for of by over under between through into '
    'about is are was were be been being have has had do does did can could will would '
    'shall should may might must not no nor all any some each every both either neither '
    'other another such only also just very too more most less least here there when where '
    'why how what whether then now still yet even much many few little there their then '
    'however therefore nevertheless instead furthermore moreover perhaps certainly likely '
    'should would could these those another otherwise rather thus since despite although'
).split())
TOKEN_RE = re.compile(r"\w+|[^\w\s]", flags=re.UNICODE)
LETTER_RE = re.compile(r'[^\W\d_]+', flags=re.UNICODE)

def shape_text(t):
    # Preserve punctuation and word LENGTHS but disguise lexical content.
    return LETTER_RE.sub(lambda m: 'X'*len(m.group()) if m.group()[0].isupper()
                         else 'x'*len(m.group()), t)

def function_text(t):
    return ' '.join(z.lower() if z.lower() in FUNCTION_WORDS else
                    ('CONTENT' if z.isalnum() else z) for z in TOKEN_RE.findall(t))

# (name, TFIDF input transformation, vectorizer arguments, relative block weight)
GRAM_SPECS = [
    ('char', str, dict(analyzer='char', ngram_range=(3,5), max_features=180_000, min_df=2, lowercase=False), 1.0),
    ('word', str, dict(analyzer='word', ngram_range=(1,3), max_features=125_000, min_df=2,
                       lowercase=False, token_pattern=r'(?u)\b\w+\b|[^\w\s]'), 0.70),
    ('shape', shape_text, dict(analyzer='char', ngram_range=(2,6), max_features=55_000, min_df=2,
                              lowercase=False), 0.55),
    ('function', function_text, dict(analyzer='word', ngram_range=(1,3), max_features=40_000, min_df=2,
                                    lowercase=False, token_pattern=r'(?u)\b\w+\b|[^\w\s]'), 0.55),
]

class TfidfSVM:
    def __init__(self, specs, C=0.5):
        self.specs, self.C = specs, C
    def fit_predict(self, xtrain, ytrain, xvalid):
        a, b = [], []
        for name, transform, opts, w in self.specs:
            vec = TfidfVectorizer(sublinear_tf=True, dtype=np.float32, **opts)
            tr = [transform(t) for t in xtrain]
            va = [transform(t) for t in xvalid]
            a.append(vec.fit_transform(tr)*w)
            b.append(vec.transform(va)*w)
        a = hstack(a, format='csr', dtype=np.float32)
        b = hstack(b, format='csr', dtype=np.float32)
        clf = LinearSVC(C=self.C, class_weight='balanced', dual='auto', max_iter=5000,
                        random_state=SEED).fit(a, ytrain)
        margins = clf.decision_function(b)
        out=np.full((len(xvalid), K), -15.0, dtype=np.float64)
        out[:,clf.classes_] = margins
        del a,b
        return out

# More lexical, complementary to the style-aware model.
PLAIN_SPECS = [
    ('char_wb', str, dict(analyzer='char_wb',ngram_range=(3,6),min_df=2,max_features=120_000), 1.0),
    ('word', str, dict(analyzer='word',ngram_range=(1,2),min_df=2,max_features=75_000,
                      token_pattern=r'(?u)\b\w+\b'), 0.65),
]

In [3]:
WORD_RE = re.compile(r"[A-Za-z]+(?:['’][A-Za-z]+)?")
SENT_RE = re.compile(r'(?<=[.!?])\s+')
PUNCT = list(string.punctuation) + list('—–−…‘’“”«»•·\n\t\r')
FUNCTION = sorted(ENGLISH_STOP_WORDS | FUNCTION_WORDS)
PATTERNS = [r'\b\w+[’\x27]\w+', r'\b\d+\b', r'\([^)]*\)', r'\*\*',
            r'\b[A-Z]{2,}\b', r'(?m)^\s*[-*•]', r'(?m)^\s*\d+[.)]',
            r'\$[^$]+\$', r'\\[a-z]+', r'\b\w+-\w+\b',
            r'\b\w+—\w+\b', r':\s', r';\s', r',\s', r'\n\s*\n']
PATTERNS=[re.compile(p) for p in PATTERNS]

def stylometry(t):
    words=WORD_RE.findall(t)
    low=[z.lower() for z in words]
    cnt=Counter(low)
    nw=max(1,len(words)); nc=max(1,len(t))
    lens=np.array([len(z) for z in words] or [0],dtype=float)
    sents=[s for s in SENT_RE.split(t) if s.strip()]
    slens=np.array([len(WORD_RE.findall(s)) for s in sents] or [0],dtype=float)
    f=[np.log1p(nc),np.log1p(nw),nc/nw,len(set(low))/nw,
       np.log1p(len(sents)),np.log1p(len(t.splitlines())),
       len(t.rstrip()) != len(t),t.startswith(('#','-','*','1.')),t.endswith(('.','!','?')),
       sum(x.isupper() for x in t)/nc,sum(x.isdigit() for x in t)/nc,
       sum(ord(x)>127 for x in t)/nc,sum(x.isspace() for x in t)/nc,
       np.mean([z[0].isupper() for z in words]) if words else 0,
       np.mean([z.isupper() for z in words]) if words else 0,
       lens.mean(),lens.std(),slens.mean(),slens.std(),
       np.mean(lens>=9),np.mean(slens<=5),
       sum(z.endswith('ly') for z in low)/nw,
       sum(z.endswith('ing') for z in low)/nw,
       sum(z.endswith('tion') for z in low)/nw]
    f+=np.quantile(lens,[0,.25,.5,.75,.9,1]).tolist()
    f+=np.quantile(slens,[0,.25,.5,.75,1]).tolist()
    f+=[sum(len(z)==k for z in words)/nw for k in range(1,16)]
    f+=[t.count(p)/nw for p in PUNCT]
    f+=[cnt[z]/nw for z in FUNCTION]
    f+=[len(p.findall(t))/nw for p in PATTERNS]
    return np.asarray(f,dtype=np.float32)

style_train=np.stack([stylometry(t) for t in texts])
style_test=np.stack([stylometry(t) for t in test_texts])
assert np.isfinite(style_train).all() and np.isfinite(style_test).all()
print('Stylometry dimensions:',style_train.shape)

def make_style_lgbm():
    return LGBMClassifier(n_estimators=320,learning_rate=0.035,num_leaves=15,
                          min_child_samples=12,colsample_bytree=0.75,
                          subsample=0.85,subsample_freq=1,reg_lambda=1.,
                          class_weight='balanced',random_state=SEED,
                          n_jobs=1,verbosity=-1)

MODEL_NAMES=('stylegram','plain_svm','style_lgb')

def fit_predict_all(train_idx, xvalid, val_style):
    # This re-fits vectorizers on training rows in EACH fold: no test leakage.
    tr=[texts[i] for i in train_idx]
    yt=y[train_idx]
    out={}
    out['stylegram']=TfidfSVM(GRAM_SPECS,C=0.5).fit_predict(tr,yt,xvalid)
    out['plain_svm']=TfidfSVM(PLAIN_SPECS,C=1.0).fit_predict(tr,yt,xvalid)
    mod=make_style_lgbm().fit(style_train[train_idx], yt)
    p=np.full((len(xvalid),K),1e-8,dtype=np.float64)
    p[:,mod.classes_]=mod.predict_proba(val_style)
    p/=p.sum(1,keepdims=True)
    out['style_lgb']=p
    return out

Stylometry dimensions: (2699, 441)


## Models
---
* Character n-gram TF-IDF + averaged SGD classifier (for lexcial and punctuation habits)
* Flat LightGBM (for 18-way stylometry)
* Hierarchical LightGBM (Estimates model-family probability) 

In [4]:

from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import balanced_accuracy_score
from scipy.special import softmax
import time

# All labels with the same prefix use the same model family.
# No provider labels from hidden data and no ID tricks.
FAMILIES=np.array([x.split('-')[0] for x in CLASSES])
FAMILY_LABELS=sorted(set(FAMILIES))
FAMILY_TO_CLASS={g:np.flatnonzero(FAMILIES==g) for g in FAMILY_LABELS}
y_family=np.array([FAMILY_LABELS.index(FAMILIES[i]) for i in y])
assert sum(len(z) for z in FAMILY_TO_CLASS.values())==K

def lgb_flat():
    return LGBMClassifier(n_estimators=100,learning_rate=0.055,
        num_leaves=10,min_child_samples=15,colsample_bytree=0.65,
        reg_lambda=2.,random_state=SEED,n_jobs=1,verbosity=-1)

def lgb_family():
    return LGBMClassifier(n_estimators=180,learning_rate=0.04,
        num_leaves=11,min_child_samples=15,random_state=SEED,
        n_jobs=1,verbosity=-1)

def lgb_within():
    return LGBMClassifier(n_estimators=150,learning_rate=0.055,
        num_leaves=9,min_child_samples=10,random_state=SEED,
        n_jobs=1,verbosity=-1)

def score_models(tr_idx,va_texts,va_style):
    # Vectorizer only fits on this fold's training data.
    tr_texts=[texts[i] for i in tr_idx]
    vec=TfidfVectorizer(analyzer='char',ngram_range=(3,5),
        max_features=90_000,min_df=3,sublinear_tf=True,dtype=np.float32)
    xtr=vec.fit_transform(tr_texts)
    xva=vec.transform(va_texts)
    clf=SGDClassifier(loss='modified_huber',alpha=1e-5,
        class_weight='balanced',average=True,max_iter=1250,
        n_jobs=1,random_state=SEED,tol=1e-4)
    clf.fit(xtr,y[tr_idx])
    margins=clf.decision_function(xva)
    char_probs=softmax(margins/0.16,axis=1).astype(np.float32)
    del xtr,xva

    # Direct 18-way stylometric prediction.
    flat=lgb_flat().fit(style_train[tr_idx],y[tr_idx])
    flat_probs=flat.predict_proba(va_style).astype(np.float32)

    # Hierarchical: P(model | text) = P(family | text) * P(model | family, text).
    fam_model=lgb_family().fit(style_train[tr_idx],y_family[tr_idx])
    coarse_probs=fam_model.predict_proba(va_style)
    hier_probs=np.zeros((len(va_texts),K),dtype=np.float32)
    for fam,class_ids in FAMILY_TO_CLASS.items():
        fam_id=FAMILY_LABELS.index(fam)
        prior=coarse_probs[:,fam_id]
        if len(class_ids)==1:
            hier_probs[:,class_ids[0]]=prior
            continue
        indices=tr_idx[np.isin(y[tr_idx],class_ids)]
        inner=lgb_within().fit(style_train[indices],y[indices])
        hier_probs[:,inner.classes_] = prior[:,None]*inner.predict_proba(va_style)
    hier_probs /= np.maximum(1e-12,hier_probs.sum(axis=1,keepdims=True))
    return {'char':char_probs, 'flat':flat_probs, 'hier':hier_probs}

# Weights were explored on this small training sample; consider them provisional.
WEIGHTS={'char':0.10,'flat':0.30,'hier':0.60}

def combine(predictions,weights=WEIGHTS):
    # Geometric averaging smooths overconfident probabilities.
    z=np.zeros_like(predictions['char'],dtype=np.float64)
    for name,weight in weights.items():
        z+=weight*np.log(np.clip(predictions[name],1e-8,1.))
    return softmax(z,axis=1)


## Cross-Validation 
---

In [5]:
RUN_CV = True 
if RUN_CV:
    
    from sklearn.metrics import confusion_matrix
    fold_oof={name:np.zeros((len(y),K),dtype=np.float32) for name in WEIGHTS}
    fold_rows=[]
    t0=time.time()
    for fold,(tr,va) in enumerate(GroupKFold(n_splits=N_FOLDS).split(texts,y,train.prompt_id)):
        output=score_models(tr,[texts[i] for i in va],style_train[va])
        for name,p in output.items():
            fold_oof[name][va]=p
        blended=combine(output)
        fold_rows.append({'fold':fold+1,'prompts':len(set(train.prompt_id.iloc[va])),
            'n':len(va),**{name:balanced_accuracy_score(y[va],output[name].argmax(1)) for name in WEIGHTS},
            'blend':balanced_accuracy_score(y[va],blended.argmax(1))})
        print('Completed fold',fold+1,'of',N_FOLDS,'in',round(time.time()-t0,1),'seconds',flush=True)
    
    print(pd.DataFrame(fold_rows).round(4).to_string(index=False))
    print('\nPooled prompt-disjoint OOF balanced accuracy:')
    for name,probs in fold_oof.items():
        print(f'{name:>7}: {balanced_accuracy_score(y,probs.argmax(1)):.4f}')
    ensemble_oof=combine(fold_oof)
    print('Ensemble:',round(balanced_accuracy_score(y,ensemble_oof.argmax(1)),4))
    print('NOTE: blend weights were explored on these folds, so the apparent gain is optimistic.')
    conf=confusion_matrix(y,ensemble_oof.argmax(1),labels=np.arange(K))
    per_class=pd.DataFrame({'model':CLASSES,'recall':np.diag(conf)/np.maximum(1,conf.sum(axis=1)),
                            'support':conf.sum(axis=1)})
    print('\nPer-class recall (worst to best):')
    print(per_class.sort_values('recall').round(3).to_string(index=False))
    print('\nMost frequent confusions:')
    conf_no_diag=conf.copy();np.fill_diagonal(conf_no_diag,0)
    for pos in np.argsort(conf_no_diag.ravel())[::-1][:12]:
        src,dest=divmod(pos,K)
        print(f'{CLASSES[src]:25s} -> {CLASSES[dest]:25s}  {conf_no_diag[src,dest]}')
else:
    print("Skipped CV to keep Kaggle submission fast. For grouped validation, set NTLLM_RUN_CV=1 before running the notebook.")

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Completed fold 1 of 5 in 22.0 seconds


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/ut

Completed fold 2 of 5 in 43.8 seconds


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/ut

...

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/ut

Completed fold 4 of 5 in 87.6 seconds


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/ut

Completed fold 5 of 5 in 109.3 seconds
 fold  prompts   n   char   flat   hier  blend
    1       30 540 0.2500 0.2759 0.2574 0.3241
    2       30 540 0.2593 0.2833 0.3037 0.3259
    3       30 540 0.2630 0.3130 0.3204 0.3370
    4       30 540 0.2833 0.2759 0.2759 0.3463
    5       30 539 0.2706 0.3003 0.3115 0.3558

Pooled prompt-disjoint OOF balanced accuracy:
   char: 0.2652
   flat: 0.2897
   hier: 0.2938
Ensemble: 0.3379
NOTE: blend weights were explored on these folds, so the apparent gain is optimistic.

Per-class recall (worst to best):
                 model  recall  support
         glm-5.3-flash   0.107      150
               glm-5.3   0.140      150
           gpt-5.6-sol   0.167      150
               kimi-k3   0.180      150
        claude-fable-5   0.215      149
               gpt-5.5   0.267      150
           qwen3.8-max   0.267      150
       claude-opus-4-8   0.273      150
         gpt-5.6-terra   0.280      150
          gpt-5.6-luna   0.313      150
      

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


## Refit on all labeled rows and Create Submission
---

In [6]:

from pathlib import Path
print(f'Fitting all {len(train)} labeled passages; predicting {len(test)} test passages ...',flush=True)
t0=time.time()
all_preds=score_models(np.arange(len(y)),test_texts,style_test)
test_prob=combine(all_preds)
test_class=test_prob.argmax(axis=1)
submission=pd.DataFrame({'id':test['id'].to_numpy(),
                         'model':np.asarray(CLASSES)[test_class]})
assert submission.columns.tolist()==['id','model']
assert submission.id.notna().all() and submission.id.is_unique
assert len(submission)==len(test)
assert submission.model.notna().all() and submission.model.isin(CLASSES).all()
output_path=Path('/kaggle/working/submission.parquet') if Path('/kaggle/working').is_dir() else Path('/mnt/data/submission.parquet')
try:
    submission.to_parquet(output_path,index=False)
    print('Saved',output_path,'in',round(time.time()-t0,1),'s')
except ImportError as e:

    csv_path=output_path.with_suffix('.csv')
    submission.to_csv(csv_path,index=False)
    print('Local environment has no Parquet engine; wrote debugging CSV:',csv_path)
    print('On Kaggle, pyarrow should be installed. Confirm submission.parquet exists before submitting.')
    print('Details:',str(e).split('\n')[0])
print(submission.to_string(index=False) if len(submission)<=5 else submission.head().to_string(index=False))


Fitting all 2699 labeled passages; predicting 3 test passages ...


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Saved /kaggle/working/submission.parquet in 24.7 s
          id            model
00353fb1c541    gpt-5.6-terra
005f5a9a6139      gpt-5.6-sol
0062a53d5487 gemini-3.7-flash


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
